In [18]:
#!/usr/bin/env python
import os
import re
import uuid
import requests
import asyncio
import nest_asyncio
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from asgiref.sync import sync_to_async
import django
import sys

# Add the project root directory to Python path
sys.path.append(os.path.abspath('../../'))

# Setup Django environment
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'umbrella.settings')
django.setup()

# Import Django models
from processes.models import ReviewTomogram, Review

# Configuration
FILE_SERVER_HOST = "https://czii-onsite.czbiohub.org"
SESSION_PATH = "krios1.processing/aretomo3/25apr21a/run001/vol001"
WEB_DIR_URL = urljoin(FILE_SERVER_HOST + "/", SESSION_PATH + "/")

# Match files like Position_1_2_Vol.mrc
MRC_FILENAME_PATTERN = re.compile(r'^Position_\d+_\d+_Vol\.mrc$')

# Apply nest_asyncio to allow nested event loops
nest_asyncio.apply()

def generate_uuid():
    return str(uuid.uuid4())

def parse_mrc_filename(filename):
    match = re.match(r'^Position_(\d+)_(\d+)_Vol\.mrc$', filename)
    if match:
        return f"Position_{match.group(1)}_{match.group(2)}"
    return None

def get_mrc_files_from_web():
    try:
        response = requests.get(WEB_DIR_URL)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # Find all rows that represent files
        file_rows = soup.find_all('tr', class_='file')

        valid_mrc_files = []

        for row in file_rows:
            # Look for span with class="name" inside the <a> tag
            name_tag = row.find('span', class_='name')
            if name_tag:
                filename = name_tag.text.strip()
                if MRC_FILENAME_PATTERN.match(filename):
                    valid_mrc_files.append(filename)

        return valid_mrc_files

    except requests.RequestException as e:
        print(f"❌ Failed to fetch MRC files from web: {e}")
        return []

@sync_to_async
def get_review():
    """Get or create a review instance"""
    review, created = Review.objects.get_or_create(
        review_id="a69fe92c524a4d068eb389cd08198c4a",
        defaults={
            'review_name': "Import Review",
            'review_type': "tomogram_quality",
            'run_id': "run001",
            'reconstruction_type': "DCTF",
            'status': "pending"
        }
    )
    return review

@sync_to_async
def create_tomogram(review, tomogram_id, position_id):
    return ReviewTomogram.objects.create(
        review=review,
        tomogram_id=tomogram_id,
        position_id=position_id,
        quality='',
        rejection_reasons=[],
        object_labels=[]
    )

@sync_to_async
def get_all_tomograms():
    return list(ReviewTomogram.objects.all())

async def process_files_async(mrc_files):
    # Get the review instance first
    review = await get_review()
    
    for filename in mrc_files:
        position_id = parse_mrc_filename(filename)
        if position_id:
            tomogram_id = generate_uuid()
            await create_tomogram(review, tomogram_id, position_id)
            print(f"✅ Created tomogram: {tomogram_id} for position {position_id}")
        else:
            print(f"❌ Skipping invalid filename: {filename}")

async def main_async():
    print(f"📂 Fetching MRC files from: {WEB_DIR_URL}")
    mrc_files = get_mrc_files_from_web()

    print(f"🔍 Found {len(mrc_files)} matching MRC files")

    await process_files_async(mrc_files)

    imported = await get_all_tomograms()
    print(f"\n📊 Total tomograms in DB: {len(imported)}")
    for tomo in imported[:5]:
        print(f"• ID: {tomo.tomogram_id}, Position: {tomo.position_id}")

def main():
    try:
        asyncio.run(main_async())
    except RuntimeError as e:
        # If we're already in an event loop (like in Jupyter), use this approach
        loop = asyncio.get_event_loop()
        loop.run_until_complete(main_async())

if __name__ == "__main__":
    main() 

📂 Fetching MRC files from: https://czii-onsite.czbiohub.org/krios1.processing/aretomo3/25apr21a/run001/vol001/
🔍 Found 29 matching MRC files
✅ Created tomogram: afcaaa20-0348-4d7c-b3ea-58667a620555 for position Position_11_2
✅ Created tomogram: 8f069f28-5c62-497d-b516-b980b76902ec for position Position_11_3
✅ Created tomogram: eececb5f-8827-4079-b080-87d1f106b919 for position Position_13_2
✅ Created tomogram: 4bd29a5b-538a-44ae-a34d-67ca89cd1930 for position Position_13_3
✅ Created tomogram: c8d9c9be-5fa8-48b5-a111-f4dad475b411 for position Position_15_2
✅ Created tomogram: 50f578e5-742a-4e03-9230-9f642e147e9e for position Position_15_3
✅ Created tomogram: 848927cc-5e78-4028-ac6a-006bfd43bff0 for position Position_15_4
✅ Created tomogram: b8234479-6ace-4e0a-a752-75110efee8c5 for position Position_17_2
✅ Created tomogram: 0d7ec08c-df9b-4759-af9d-5205be3faeab for position Position_17_3
✅ Created tomogram: 3a8d4eed-caa7-4eeb-9aec-79db18d1062f for position Position_18_2
✅ Created tomogram: